In [1]:
import torch 
import torch.nn as nn 
import torch.nn.init as init
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
transform = transforms.Compose([
    transforms.ToTensor()]
)
train_set = datasets.FashionMNIST(
    root = 'data',
    download = True,
    train = True,
    transform = transform
)
test_set = datasets.FashionMNIST(
    root = 'data',
    download = True,
    train = False,
    transform = transform
)

trainloader = DataLoader(
    train_set,
    batch_size = 256,
    num_workers = 4,
    shuffle = True
)
testloader = DataLoader(
    test_set,
    batch_size = 256,
    shuffle = True,
    num_workers = 4
)

In [3]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv_layer1 = nn.Sequential(
            nn.Conv2d(in_channels = 1, out_channels = 64, kernel_size = 3, padding = 'same'),
            nn.Sigmoid(),
            nn.MaxPool2d(2, 2)
        )
        self.conv_layer2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding = 'same'),
            nn.Sigmoid(),
            nn.MaxPool2d(2, 2)
        )
        # Flatten
        self.flatten = nn.Flatten()

        # Fully Connected Layer 1
        self.fc_layer1 = nn.Sequential(
            nn.Linear(128 * 7 * 7, 512),
            nn.Sigmoid()
        )
        # Fully Connected Layer 2
        self.fc_layer2 = nn.Sequential(
            nn.Linear(512, 10)
        )
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    init.zeros_(m.bias)
    def forward(self, x):
        x = self.conv_layer1(x)
        x = self.conv_layer2(x)
        x = self.flatten(x)
        x = self.fc_layer1(x)
        x = self.fc_layer2(x)
        return x

In [ ]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device {device}')

#Initialize model
model = CNN().to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr = 1e-3)

losses = []
# Train model
epochs = 100
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0

    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)

        # Forward propagation
        outputs = model(images)

        # Compute loss
        loss = criterion(outputs, labels)

        # Backward propagation
        optimizer.zero_grad()
        loss.backward()

        # Update parameters
        optimizer.step()

        train_loss += loss.item() * labels.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    train_losses = train_loss / total
    losses.append(train_losses)
    train_accuracy = correct / total * 100

    print(
        f'Epoch : [{epoch + 1} / {epochs}] '
        f'Loss: {train_loss:.4f} '
        f'Accuracy: {train_accuracy:.2f}%'
    )


Device mps
Epoch : [1 / 100] Loss: 105031.6858 Accuracy: 38.55%
Epoch : [2 / 100] Loss: 36640.1986 Accuracy: 77.15%
Epoch : [3 / 100] Loss: 29816.6094 Accuracy: 81.67%
Epoch : [4 / 100] Loss: 26246.0611 Accuracy: 83.89%
Epoch : [5 / 100] Loss: 24012.1491 Accuracy: 85.21%
Epoch : [6 / 100] Loss: 22399.0284 Accuracy: 86.17%
Epoch : [7 / 100] Loss: 21184.8452 Accuracy: 87.06%
Epoch : [8 / 100] Loss: 20266.1009 Accuracy: 87.52%
Epoch : [9 / 100] Loss: 19081.7565 Accuracy: 88.25%
Epoch : [10 / 100] Loss: 18454.7501 Accuracy: 88.63%
Epoch : [11 / 100] Loss: 17813.4590 Accuracy: 89.09%
Epoch : [12 / 100] Loss: 17315.3589 Accuracy: 89.29%
Epoch : [13 / 100] Loss: 16707.4554 Accuracy: 89.61%
Epoch : [14 / 100] Loss: 16076.1145 Accuracy: 90.06%
Epoch : [15 / 100] Loss: 15750.9726 Accuracy: 90.19%
Epoch : [16 / 100] Loss: 15383.6056 Accuracy: 90.48%
Epoch : [17 / 100] Loss: 14661.1848 Accuracy: 90.83%
Epoch : [18 / 100] Loss: 14360.4774 Accuracy: 91.08%
Epoch : [19 / 100] Loss: 13905.9295 Accurac

In [10]:
# Evaluate model
model.eval()
correct = 0
total = 0
test_loss = 0.0
with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        test_loss += loss.item() * labels.size(0)

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)

        correct += (predicted == labels).sum().item()
    test_accuracy = correct / total * 100
    test_loss = test_loss / total
    print(f'Test Loss: {test_loss:.4f}')
    print(f"Test Accuracy: {test_accuracy:.2f}%")


Test Loss: 0.3915
Test Accuracy: 91.45%
